<a href="https://colab.research.google.com/github/febriyansyah-id/COLLAB/blob/main/notebooks/NLP-assignment-P03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Assignment P03
## Text Preprocessing untuk NLP

Notebook ini digunakan di Google Colab untuk mempraktikkan preprocessing teks menggunakan NLTK dan Stanza.

Pipeline yang dipelajari: normalisasi, punctuation removal, tokenisasi, stopword removal, stemming, lemmatisasi, POS tagging, dependency parsing, dan NER.

## Tujuan

1. Membuat fungsi `preprocess_text` dengan NLTK.
2. Menampilkan token, lemma, POS, dependency, dan NER dengan Stanza.
3. Mencoba pemrosesan multilingual.
4. Membandingkan hasil preprocessing NLTK dan Stanza.
5. Menjelaskan konsistensi preprocessing dan pemilihan tools.

## 1. Instalasi dan resource Google Colab

In [1]:
%pip install -q nltk stanza scikit-learn Sastrawi flask

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 16.3 MB/s eta 0:00:00


In [2]:
import string
import nltk

resources = [
    ('tokenizers/punkt', 'punkt'),
    ('tokenizers/punkt_tab', 'punkt_tab'),
    ('corpora/stopwords', 'stopwords'),
    ('corpora/wordnet', 'wordnet'),
    ('corpora/omw-1.4', 'omw-1.4'),
    ('taggers/averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger_eng'),
]
for path, package in resources:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(package, quiet=True)
print('Resource NLTK siap digunakan.')

Resource NLTK siap digunakan.


## 2. Preprocessing dengan NLTK

Fungsi berikut mengembalikan hasil setiap tahap dalam bentuk dictionary. Contoh menggunakan teks bahasa Inggris agar resource NLTK standar dapat digunakan.

In [3]:
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

def preprocess_text(text):
    normalized = text.lower()
    without_punctuation = normalized.translate(
        str.maketrans('', '', string.punctuation)
    )
    tokens = word_tokenize(without_punctuation)
    stop_words = set(stopwords.words('english'))
    filtered = [t for t in tokens if t.isalnum() and t not in stop_words]
    stemmer = PorterStemmer()
    stemmed = [stemmer.stem(t) for t in filtered]
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(t) for t in filtered]
    tagged = pos_tag(filtered)
    return {
        'original_text': text,
        'normalized_text': normalized,
        'without_punctuation': without_punctuation,
        'tokens': tokens,
        'filtered_tokens': filtered,
        'stemmed_tokens': stemmed,
        'lemmatized_tokens': lemmatized,
        'pos_tagged_tokens': tagged,
    }

text = 'Natural Language Processing is a field of artificial intelligence.'
nltk_result = preprocess_text(text)
for stage, value in nltk_result.items():
    print(f'{stage}: {value}')

original_text: Natural Language Processing is a field of artificial intelligence.
normalized_text: natural language processing is a field of artificial intelligence.
without_punctuation: natural language processing is a field of artificial intelligence
tokens: ['natural', 'language', 'processing', 'is', 'a', 'field', 'of', 'artificial', 'intelligence']
filtered_tokens: ['natural', 'language', 'processing', 'field', 'artificial', 'intelligence']
stemmed_tokens: ['natur', 'languag', 'process', 'field', 'artifici', 'intellig']
lemmatized_tokens: ['natural', 'language', 'processing', 'field', 'artificial', 'intelligence']
pos_tagged_tokens: [('natural', 'JJ'), ('language', 'NN'), ('processing', 'NN'), ('field', 'NN'), ('artificial', 'JJ'), ('intelligence', 'NN')]


## 3. Preprocessing dengan Stanza

Stanza menyediakan pipeline neural untuk tokenisasi, lemma, POS tagging, dependency parsing, dan NER. Model bahasa perlu diunduh satu kali di runtime Colab.

In [4]:
import stanza
stanza.download('en', verbose=False)
nlp_en = stanza.Pipeline('en', processors='tokenize,mwt,pos,lemma,depparse,ner', verbose=False)
doc = nlp_en('Sajarwo Anggai teaches Advanced NLP at Pamulang University.')

for sentence in doc.sentences:
    for word in sentence.words:
        print({
            'text': word.text,
            'lemma': word.lemma,
            'upos': word.upos,
            'head': word.head,
            'dependency': word.deprel,
        })
    print('NER:', [(ent.text, ent.type) for ent in sentence.ents])

models/default.zip: reconstructing file:   0%|          |  0.00B /  525MB            

models/default.zip: downloading bytes:           |  0.00B            

{'text': 'Sajarwo', 'lemma': 'Sajarwo', 'upos': 'PROPN', 'head': 3, 'dependency': 'nsubj'}
{'text': 'Anggai', 'lemma': 'Anggai', 'upos': 'PROPN', 'head': 1, 'dependency': 'flat'}
{'text': 'teaches', 'lemma': 'teach', 'upos': 'VERB', 'head': 0, 'dependency': 'root'}
{'text': 'Advanced', 'lemma': 'Advanced', 'upos': 'ADJ', 'head': 5, 'dependency': 'amod'}
{'text': 'NLP', 'lemma': 'NLP', 'upos': 'PROPN', 'head': 3, 'dependency': 'obj'}
{'text': 'at', 'lemma': 'at', 'upos': 'ADP', 'head': 8, 'dependency': 'case'}
{'text': 'Pamulang', 'lemma': 'Pamulang', 'upos': 'PROPN', 'head': 8, 'dependency': 'compound'}
{'text': 'University', 'lemma': 'University', 'upos': 'PROPN', 'head': 3, 'dependency': 'obl'}
{'text': '.', 'lemma': '.', 'upos': 'PUNCT', 'head': 3, 'dependency': 'punct'}
NER: [('Sajarwo Anggai', 'PERSON'), ('Advanced NLP', 'ORG'), ('Pamulang University', 'ORG')]


## 4. Multilingual processing

Gunakan model yang sesuai dengan bahasa dokumen. Contoh berikut memproses bahasa Inggris, Indonesia, dan Spanyol secara terpisah.

In [5]:
languages = {
    'en': 'Natural Language Processing analyzes text.',
    'id': 'Pemrosesan bahasa alami menganalisis teks.',
    'es': 'El procesamiento del lenguaje natural analiza textos.',
}
for language in languages:
    stanza.download(language, verbose=False)

pipelines = {
    language: stanza.Pipeline(language, processors='tokenize,pos,lemma', verbose=False)
    for language in languages
}
for language, sentence in languages.items():
    doc = pipelines[language](sentence)
    tokens = [(word.text, word.lemma, word.upos) for s in doc.sentences for word in s.words]
    print(language, tokens)

models/default.zip: reconstructing file:   0%|          |  0.00B /  394MB            

models/default.zip: downloading bytes:           |  0.00B            

models/default.zip: reconstructing file:   0%|          |  0.00B /  640MB            

models/default.zip: downloading bytes:           |  0.00B            

en [('Natural', 'Natural', 'ADJ'), ('Language', 'language', 'NOUN'), ('Processing', 'processing', 'NOUN'), ('analyzes', 'analyze', 'VERB'), ('text', 'text', 'NOUN'), ('.', '.', 'PUNCT')]
id [('Pemrosesan', 'proses', 'NOUN'), ('bahasa', 'bahasa', 'NOUN'), ('alami', 'alami', 'ADJ'), ('menganalisis', 'analisis', 'VERB'), ('teks', 'teks', 'NOUN'), ('.', '.', 'PUNCT')]
es [('El', 'el', 'DET'), ('procesamiento', 'procesamiento', 'NOUN'), ('de', 'de', 'ADP'), ('el', 'el', 'DET'), ('lenguaje', 'lenguaje', 'NOUN'), ('natural', 'natural', 'ADJ'), ('analiza', 'analizar', 'VERB'), ('textos', 'texto', 'NOUN'), ('.', '.', 'PUNCT')]


## 5. Perbandingan NLTK dan Stanza

NLTK cocok untuk pembelajaran dan preprocessing klasik. Stanza menyediakan pipeline neural yang lebih lengkap untuk lemma, POS, dependency, dan NER. Hasil dapat berbeda karena tokenizer, model, tagset, dan resource bahasa yang digunakan.

In [6]:
comparison = {
    'NLTK': {
        'strength': 'ringan, modular, mudah dipelajari',
        'output': list(nltk_result.keys()),
    },
    'Stanza': {
        'strength': 'pipeline neural dan anotasi linguistik lengkap',
        'output': ['token', 'lemma', 'POS', 'dependency', 'NER'],
    },
}
comparison

{'NLTK': {'strength': 'ringan, modular, mudah dipelajari',
  'output': ['original_text',
   'normalized_text',
   'without_punctuation',
   'tokens',
   'filtered_tokens',
   'stemmed_tokens',
   'lemmatized_tokens',
   'pos_tagged_tokens']},
 'Stanza': {'strength': 'pipeline neural dan anotasi linguistik lengkap',
  'output': ['token', 'lemma', 'POS', 'dependency', 'NER']}}

## 6. Contoh API Flask

File `tugas/app.py` pada materi menyediakan endpoint `POST /process`. Dalam tugas Colab ini, fungsi preprocessing sudah diuji langsung. API dapat dijalankan sebagai pengembangan lanjutan dengan payload:

```json
{"text": "Natural Language Processing is useful."}
```

## 7. Analisis dan kesimpulan

Jawab pertanyaan berikut berdasarkan output notebook:

1. Apa dampak lowercasing, punctuation removal, dan stopword removal?
2. Mengapa stemming dapat menghasilkan kata yang tidak baku?
3. Apa perbedaan hasil NLTK dan Stanza?
4. Mengapa model bahasa harus disesuaikan dengan dokumen?
5. Apa risiko jika preprocessing training dan testing tidak konsisten?

Tuliskan kesimpulan 3–5 paragraf dan sertakan contoh input-output yang paling penting.